In [ ]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

In [ ]:

chroma_client = chromadb.Client()

# Use OpenAI for embeddings (or swap for HuggingFace)
embedding_fn = OpenAIEmbeddingFunction(api_key="your-key-here")

# Create a collection
collection = chroma_client.create_collection(
    name="nl2sql_examples",
    embedding_function=embedding_fn
)

# Add examples
collection.add(
    documents=[
        "Show the top 5 customers by revenue",
        "Total revenue per region",
        "Orders from last 30 days"
    ],
    metadatas=[
        {"sql": "SELECT name FROM customers ORDER BY revenue DESC LIMIT 5"},
        {"sql": "SELECT region, SUM(revenue) FROM customers GROUP BY region"},
        {"sql": "SELECT * FROM orders WHERE order_date >= CURRENT_DATE - INTERVAL '30 days'"}
    ],
    ids=["1", "2", "3"]
)


In [ ]:
# To add DDl 
collection.add(
    documents=["CREATE TABLE customers (...);", "CREATE TABLE orders (...);"],
    metadatas=[{"type": "ddl"}, {"type": "ddl"}],
    ids=["ddl1", "ddl2"]
)

In [ ]:
from dspy.retrieve.api import BaseRetriever

class ChromaRetriever(BaseRetriever):
    def __init__(self, collection, top_k=2):
        self.collection = collection
        self.top_k = top_k

    def forward(self, question, **kwargs):
        results = self.collection.query(query_texts=[question], n_results=self.top_k)
        return results['documents'][0]  # Return list of top documents


In [ ]:
import dspy

retriever = ChromaRetriever(collection)

class SQLWithContext(dspy.Signature):
    question = dspy.InputField()
    context = dspy.InputField(desc="Relevant examples or docs from Chroma")
    sql = dspy.OutputField()

class SQLAgentWithRAG(dspy.Module):
    def __init__(self):
        super().__init__()
        self.retrieve = dspy.Retrieve(k=2, retriever=retriever)
        self.generator = dspy.ChainOfThought(SQLWithContext)

    def forward(self, question):
        docs = self.retrieve(question=question).passages
        return self.generator(question=question, context="\n".join(docs))


In [ ]:
dspy.settings.configure(lm=dspy.OpenAI(model="gpt-4"))

agent = SQLAgentWithRAG()
response = agent("Which customer has the highest revenue?")
print(response.sql)
